# Лабораторная работа №2

Вариант:
- Задача 4
- Задание 2

Выполнили студенты 25ИАД:
- Ерёменко Данил
- Кудасов Максим

## Формулировка задания

Для выполнения лабораторной работы рассмотрим функция из задачи 4:

$
f(x) = \cfrac{\rho}{2} \| Ax - b \|^2 + \sum_{i=1}^{m} \log\left(2 \cosh \bigl(u_i^\top (x-a) \bigr) \right),
$

где

- $x \in \mathbb{R}^n$ — переменная для оптимизации,
- $a \in \mathbb{R}^n$,
- $b \in \mathbb{R}^m$,
- $A \in \mathbb{R}^{m\times n}$,
- $u_i \in \mathbb{R}^n$,
- $\rho \ge 0$,
- $m = \frac{n}{5}$.

Дополнительно накладываются ограничения вида

$$
-R \le x_i \le R, \qquad i=1,\dots,n.
$$

То есть допустимое множество имеет вид

$$
Q=[-R,R]^n.
$$

Для данной функции требуется выполнить несколько заданий. Среди них есть теоретическое и практическое.

## Теоретическая часть

Теоретические задания будут выполнены в текущем разделе. Список заданий следующий:
1. Проверка выпуклости множества допустимых точек
2. Градиентный спуск с постоянным шагом с учетом ограничений
3. Представление функции в виде суммы $f(x)=\sum_{j=1}^{M}\phi_j(x)$
4. Построение стохастического градиента
5. Проверка несмещённости стохастического градиента
7. Проверка ограниченности дисперсии
8. Подготовить постановку для минибатчинга и дальнейших экспериментов

### 1. Проверка выпуклости множества допустимых точек

Нам требуется проверить, что множество

$$
Q=[-R,R]^n
$$

является выпуклым.

По определению, множество $Q$ выпукло, если для любых $x,y \in Q$ и любого $\lambda \in [0,1]$ выполнено:

$$
\lambda x + (1-\lambda)y \in Q.
$$

Так как $Q$ — это декартово произведение отрезков $[-R,R]$, достаточно проверить это покоординатно.

Если $x,y \in Q$, то для каждой координаты выполняется

$$
-R \le x_i \le R, \qquad -R \le y_i \le R.
$$

Тогда для любой $\lambda \in [0,1]$:

$$
\lambda x_i + (1-\lambda)y_i
$$

является выпуклой комбинацией чисел из отрезка $[-R,R]$, а значит, тоже принадлежит этому отрезку:

$$
-R \le \lambda x_i + (1-\lambda)y_i \le R.
$$

Следовательно, каждая координата вектора $\lambda x + (1-\lambda)y$ принадлежит $[-R,R]$, а значит,

$$
\lambda x + (1-\lambda)y \in [-R,R]^n = Q.
$$

### 2. Градиентный спуск с постоянным шагом с учетом ограничений

В задании требуется реализовать градиентный спуск с постоянным шагом. Учитывая, что рассматриваемая задача решается не на всей $\mathbb{R}^n$, а на выпуклом множестве

$$
Q=[-R,R]^n,
$$

мы будем использовать **проецированный градиентный спуск**:

$$
x_{k+1}=\Pi_Q\bigl(x_k-h\nabla f(x_k)\bigr),
$$

где $\Pi_Q$ — евклидова проекция на множество $Q$.

Для нашего множества $Q=[-R,R]^n$  проекция получается покоординатным отсечением значений за пределами интервала.

То есть для любого вектора $z \in \mathbb{R}^n$:

$$
(\Pi_Q(z))_i=\min\{R,\max\{-R,z_i\}\}.
$$

На практике это означает, что после градиентного шага мы просто "обрезаем" каждую координату, если она вышла за границы $[-R,R]$.

### 3. Представление функции в виде суммы $f(x)=\sum_{j=1}^{M}\phi_j(x)$

Для построения стохастического градиента по условию задания нам нужно представить функцию в виде суммы простых слагаемых.

Для нашей задачи можно выбрать следующее разложение:

$$
f(x)=\phi_1(x)+\sum_{i=1}^{m}\phi_{i+1}(x),
$$

где

$$
\phi_1(x)=\frac{\rho}{2}\|Ax-b\|^2,
$$

а для $i=1,\dots,m$:

$$
\phi_{i+1}(x)=\log\left(2\cosh\bigl(u_i^\top(x-a)\bigr)\right).
$$

Тогда общее число слагаемых равно

$$
M=m+1.
$$

Такое разложение удобно тем, что все слагаемые имеют простой градиент, и функцию можно записать как сумму "элементарных" частей.

#### Градиенты отдельных слагаемых

Для первого слагаемого:

$$
\nabla \phi_1(x)=\rho A^\top(Ax-b).
$$

Для остальных слагаемых, если $j=i+1$, $i=1,\dots,m$:

$$
\nabla \phi_j(x)=\tanh\bigl(u_i^\top(x-a)\bigr)\,u_i.
$$

Таким образом, полный градиент действительно представляется как сумма градиентов отдельных $\phi_j(x)$:

$$
\nabla f(x)=\sum_{j=1}^{M}\nabla \phi_j(x).
$$

### 4. Построение стохастического градиента

Теперь, следуя условию задания, на каждой итерации выбираем случайный индекс

$$
\xi_k \sim \mathrm{Unif}\{1,\dots,M\},
$$

независимо по $k$, и вместо полного градиента используем стохастический градиент

$$
g(x_k,\xi_k)=M\nabla \phi_{\xi_k}(x_k).
$$

В нашем случае это означает следующее:

- если выбран индекс $\xi_k=1$, то
  $$
  g(x_k,\xi_k)=M\,\rho A^\top(Ax_k-b);
  $$

- если выбран индекс $\xi_k=i+1$, где $i=1,\dots,m$, то
  $$
  g(x_k,\xi_k)=M\,\tanh\bigl(u_i^\top(x_k-a)\bigr)\,u_i.
  $$

### 5. Проверка несмещённости стохастического градиента

Покажем, что такой стохастический градиент является несмещённым.

По определению математического ожидания:

$$
\mathbb{E}_{\xi_k}[g(x_k,\xi_k)]
=
\sum_{j=1}^{M}\mathbb{P}(\xi_k=j)\, M\nabla \phi_j(x_k).
$$

Так как индекс выбирается равновероятно,

$$
\mathbb{P}(\xi_k=j)=\frac{1}{M}.
$$

Подставим это:

$$
\mathbb{E}_{\xi_k}[g(x_k,\xi_k)]
=
\sum_{j=1}^{M}\frac{1}{M} \cdot M \nabla \phi_j(x_k)
=
\sum_{j=1}^{M}\nabla \phi_j(x_k).
$$

Но по построению

$$
\sum_{j=1}^{M}\nabla \phi_j(x_k)=\nabla f(x_k).
$$

Следовательно,

$$
\mathbb{E}_{\xi_k}[g(x_k,\xi_k)]=\nabla f(x_k).
$$

То есть построенный стохастический градиент является **несмещённым**.

### 8. Ограниченность дисперсии

Проверим дисперсию полученного стохастического градиента на ограниченность.

Так как множество допустимых точек ограничено:

$$
x \in Q=[-R,R]^n,
$$

то все координаты $x$ ограничены по модулю. Следовательно, ограничен и вектор $(x-a)$.

#### Для лог-кош слагаемых

Так как функция $\tanh(z)$ ограничена:

$$
|\tanh(z)| \le 1,
$$

то для любого $i$:

$$
\|\nabla \phi_{i+1}(x)\|
=
\left\|\tanh\bigl(u_i^\top(x-a)\bigr)u_i\right\|
\le
\|u_i\|.
$$

#### Для квадратичного слагаемого

Для первого слагаемого имеем

$$
\nabla \phi_1(x)=\rho A^\top(Ax-b).
$$

Так как $x$ пробегает ограниченное множество $Q$, вектор $Ax-b$ также пробегает ограниченное множество. Следовательно, и норма

$$
\|\rho A^\top(Ax-b)\|
$$

ограничена сверху некоторой константой.

Таким образом, существует константа $G>0$ такая, что для всех $j=1,\dots,M$ и всех $x \in Q$:

$$
\|\nabla \phi_j(x)\| \le G.
$$

Тогда для стохастического градиента

$$
g(x,\xi)=M\nabla \phi_\xi(x)
$$

получаем

$$
\|g(x,\xi)\| \le MG,
$$

а значит и второй момент ограничен:

$$
\mathbb{E}\|g(x,\xi)\|^2 \le M^2 G^2.
$$

Следовательно, дисперсия стохастического градиента также ограничена.

### 9. Минибатчинг

По условию задания для методов со стохастическим градиентом требуется также рассмотреть минибатчинг.

Если на $k$-й итерации выбираются независимые индексы

$$
\xi_k^{(1)},\dots,\xi_k^{(b)} \sim \mathrm{Unif}\{1,\dots,M\},
$$

то батчированная оценка стохастического градиента имеет вид

$$
g_b(x_k)=\frac{1}{b}\sum_{r=1}^{b} g(x_k,\xi_k^{(r)}).
$$

Так как каждое $g(x_k,\xi_k^{(r)})$ является несмещённой оценкой полного градиента, среднее по батчу также остаётся несмещённым:

$$
\mathbb{E}[g_b(x_k)] = \nabla f(x_k).
$$

При этом дисперсия уменьшается с ростом размера батча $b$, что и является основной причиной, по которой минибатчинг может улучшать поведение метода на практике.

### Собственный способ уменьшения дисперсии

В качестве собственного метода уменьшения дисперсии будем использовать контрольную вариату типа SVRG.

Пусть функция представлена в виде

$$
f(x)=\sum_{j=1}^{M}\phi_j(x).
$$

Выберем опорную точку $\tilde x$ и вычислим в ней полный градиент $\nabla f(\tilde x)$. Тогда на последующих итерациях вместо обычного стохастического градиента используем оценку

$$
g_{\text{SVRG}}(x,\xi)
=
M\nabla \phi_\xi(x)-M\nabla \phi_\xi(\tilde x)+\nabla f(\tilde x),
\qquad
\xi \sim \mathrm{Unif}\{1,\dots,M\}.
$$

Такая оценка остаётся несмещённой:

$$
\mathbb{E}_\xi[g_{\text{SVRG}}(x,\xi)] = \nabla f(x),
$$

но её дисперсия обычно существенно меньше, чем у обычного стохастического градиента, особенно если $x$ находится близко к $\tilde x$.

### 10. Какие методы будем реализовывать на практике

С учётом всего вышесказанного, в практической части будем использовать следующие базовые алгоритмы:

1. **Проецированный градиентный спуск** с постоянным шагом:
   $$
   x_{k+1}=\Pi_Q\bigl(x_k-h\nabla f(x_k)\bigr).
   $$

2. **Проецированный стохастический градиентный спуск**:
   $$
   x_{k+1}=\Pi_Q\bigl(x_k-h_k g(x_k,\xi_k)\bigr).
   $$

3. **Минибатч-версию стохастического градиента**:
   $$
   x_{k+1}=\Pi_Q\bigl(x_k-h_k g_b(x_k)\bigr).
   $$

Далее на этой базе будут сравниваться:
- убывающие шаги,
- собственный способ уменьшения дисперсии,
- минибатчинг,
- комбинации методов уменьшения дисперсии,
- сравнение с одним из адаптивных методов из первой лабораторной работы.

### Собственная модель стохастического градиента

В качестве собственной модели стохастического градиента будем использовать следующую идею: квадратичную часть вычислять точно, а случайность вводить только в сумму лог-кош слагаемых.

Представим функцию в виде

$$
f(x)=\phi_0(x)+\sum_{i=1}^{m}\phi_i(x),
$$

где

$$
\phi_0(x)=\frac{\rho}{2}\|Ax-b\|^2,
\qquad
\phi_i(x)=\log\left(2\cosh\bigl(u_i^\top(x-a)\bigr)\right).
$$

Тогда определим стохастический градиент как

$$
g(x,\xi)=\nabla \phi_0(x)+m\,\nabla \phi_\xi(x),
\qquad
\xi \sim \mathrm{Unif}\{1,\dots,m\}.
$$

То есть

$$
g(x,\xi)=\rho A^\top(Ax-b)+m\,\tanh\bigl(u_\xi^\top(x-a)\bigr)\,u_\xi.
$$

Проверим несмещённость:

$$
\mathbb{E}_\xi[g(x,\xi)]
=
\rho A^\top(Ax-b)+
m \cdot \frac{1}{m}\sum_{i=1}^{m}\nabla \phi_i(x)
=
\nabla f(x).
$$

Следовательно, оценка является несмещённой.

Так как множество допустимых точек ограничено, а функция $\tanh$ ограничена по модулю единицей, существует константа $C>0$, такая что

$$
\|g(x,\xi)\| \le C
$$

для всех $x \in Q$. Поэтому второй момент, а значит и дисперсия такого стохастического градиента, ограничены.

### 11. Важное замечание про $\mu$

В условии задания для убывающих шагов предлагаются формулы

$$
h_k=\frac{2}{\mu(k+1)},
\qquad
h_k=\frac{2}{\mu\sqrt{k+1}}.
$$

Однако для нашей функции без дополнительной регуляризации глобальная сильная выпуклость, вообще говоря, может отсутствовать. Это уже наблюдалось в первой лабораторной работе: при $m=\frac{n}{5}<n$ естественная нижняя оценка на $\mu$ часто оказывалась равной нулю.

Поэтому на практике здесь есть два варианта:

1. либо ввести дополнительную регуляризацию и тем самым обеспечить $\mu>0$;
2. либо использовать некоторую малую положительную константу вместо формального $\mu$, понимая, что это уже эвристическая настройка шага.

Этот момент важно отдельно прокомментировать в отчёте.

## Практическая часть

Список заданий для выполнения практической части следующий:

1. Для каждого $n \in \{10, 20, \dots, 100\}$ сгенерировать 100 тестовых примеров и найти $x^*$ с помощью CVXPY (или любого другого сольвера). Проверить условие оптимальности.

2. Для каждого $n \in \{10, 20, \dots, 100\}$ и каждого тестового примера сгенерировать 100 начальных точек. Для точности $\varepsilon = 0.01$ реализовать методы:
   1.  Градиентный спуск с постоянным шагом $\gamma = \frac{1}{L}$.
   2.  Градиентный спуск с постоянным шагом $\gamma = \frac{1}{2L}$.

3. В качестве результата работы методов:
   1. Построить график траектории одного из методов, при $n = 2$ (если есть $m$, то $m = 1$).
   2. Для каждого $n \in \{10, 20, \dots, 100\}$ подсчитать среднее время работы метода и среднее число итераций (усреднение по всем начальным точкам и по всем тестовым примерам); сравнить с теоретическими оценками.
   3. Для одного тестового примера при $n = 10$ и нескольких различных начальных точек построить зависимость точности от числа итераций.
   4. Добавить регуляризацию Тихонова ($f_{\lambda}(x) = f(x) + \frac{\lambda}{2} \|x\|^2$) и повторить сравнение. Сравнить поведение методов при $\mu = 0$ с добавленной регуляризацией и при $\mu > 0$ без регуляризации. Как меняется сходимость метода при добавлении регуляризации при $\mu > 0$?
   5. Добавить адаптивный подбор шага и повторить сравнение.
   6. Для адаптивных методов сравнить глобальную константу $L$ со средним по траектории значением $\frac{1}{N} \sum_{k=1}^N L_k$; оценить среднее число итераций внутреннего цикла подбора шага.

4. Сравнить методы между собой и объяснить результаты. Какой метод сходится быстрее всего (для $\mu = 0$ и $\mu > 0$)?

In [ ]:
import numpy as np
import cvxpy as cp
import matplotlib.pyplot as plt
import time
import pandas as pd

from dataclasses import dataclass

In [ ]:
class Task:
    def __init__(self, n, m=None, R=5.0):
        self.n = n
        self.m = max(1, n // 5) if m is None else m
        self.R = R

        self.A = np.random.randn(self.m, self.n)
        self.b = np.random.randn(self.m)
        self.a = np.random.randn(self.n)
        self.U = np.random.randn(self.n, self.m)
        self.rho = np.random.uniform(0.1, 1.0)

    def project(self, X):
        return np.clip(X, -self.R, self.R)

    def f(self, X, lam=0.0):
        is_single = (X.ndim == 1)

        b = self.b if is_single else self.b[:, None]
        a = self.a if is_single else self.a[:, None]

        norm_part = (self.rho / 2) * np.sum((self.A @ X - b) ** 2, axis=0)
        z = self.U.T @ (X - a)
        log_cosh = np.sum(np.logaddexp(z, -z), axis=0)

        out = norm_part + log_cosh + (lam / 2) * np.sum(X ** 2, axis=0)
        return float(out) if is_single else out

    def grad(self, X, lam=0.0):
        is_single = (X.ndim == 1)

        b = self.b if is_single else self.b[:, None]
        a = self.a if is_single else self.a[:, None]

        g_norm = self.rho * self.A.T @ (self.A @ X - b)
        g_log = self.U @ np.tanh(self.U.T @ (X - a))
        return g_norm + g_log + lam * X

    def L(self, lam=0.0):
        L_A = self.rho * np.linalg.norm(self.A.T @ self.A, ord=2)
        L_U = np.linalg.norm(self.U @ self.U.T, ord=2)
        return L_A + L_U + lam

    # ---- component gradients for stochastic methods ----

    def grad_phi0(self, x, lam=0.0):
        """
        Gradient of quadratic component:
            phi_0(x) = (rho/2)||Ax-b||^2 + (lam/2)||x||^2
        """
        return self.rho * self.A.T @ (self.A @ x - self.b) + lam * x

    def grad_phi_i(self, x, i):
        """
        Gradient of i-th log-cosh component:
            phi_i(x) = log(2 cosh(u_i^T (x-a))),  i = 0,...,m-1
        """
        u_i = self.U[:, i]
        return np.tanh(u_i @ (x - self.a)) * u_i

    def num_components_baseline(self):
        """
        Baseline decomposition:
            f(x) = phi_0(x) + sum_{i=0}^{m-1} phi_i(x)
        total number of components = m + 1
        """
        return self.m + 1

In [ ]:
def projected_gd(task, X0, gamma, lam=0.0, eps=1e-2, max_iter=5000, keep_hist=False):
    is_single = (X0.ndim == 1)

    X = task.project(X0.copy())
    if is_single:
        X = X.reshape(-1, 1)

    num_starts = X.shape[1]
    iters = np.zeros(num_starts, dtype=int)
    active = np.ones(num_starts, dtype=bool)

    hist_X = [X.copy()] if keep_hist else None
    hist_f = [task.f(X, lam)] if keep_hist else None
    hist_grad = [np.linalg.norm(task.grad(X, lam), axis=0)] if keep_hist else None

    for _ in range(max_iter):
        if not np.any(active):
            break

        active_idx = np.where(active)[0]
        X_act = X[:, active_idx]
        g = task.grad(X_act, lam)

        grad_norms = np.linalg.norm(g, axis=0)
        still_active = grad_norms >= eps

        converged_idx = active_idx[~still_active]
        active[converged_idx] = False

        active_idx = active_idx[still_active]
        if active_idx.size > 0:
            g_active = g[:, still_active]
            X[:, active_idx] = task.project(X[:, active_idx] - gamma * g_active)
            iters[active_idx] += 1

        if keep_hist:
            hist_X.append(X.copy())
            hist_f.append(task.f(X, lam))
            hist_grad.append(np.linalg.norm(task.grad(X, lam), axis=0))

    if is_single:
        if keep_hist:
            return (
                X[:, 0],
                np.array(hist_X)[:, :, 0],
                np.array(hist_f)[:, 0],
                np.array(hist_grad)[:, 0],
                int(iters[0]),
            )
        return X[:, 0], int(iters[0])

    if keep_hist:
        return X, np.array(hist_X), np.array(hist_f), np.array(hist_grad), iters
    return X, iters

In [ ]:
def baseline_sgd_grad(task, x, lam=0.0):
    """
    Baseline model from the assignment:
        g(x, xi) = M * grad(phi_xi(x)),
        xi ~ Unif({0,1,...,m}),
        M = m + 1

    Here:
      xi = 0          -> quadratic part (+ optional lam regularization)
      xi = 1,...,m    -> corresponding log-cosh term
    """
    M = task.num_components_baseline()
    xi = np.random.randint(0, M)

    if xi == 0:
        g = M * task.grad_phi0(x, lam=lam)
    else:
        g = M * task.grad_phi_i(x, xi - 1)

    return g, xi

In [ ]:
def baseline_sgd_grad_minibatch(task, x, batch_size, lam=0.0):
    M = task.num_components_baseline()
    g = np.zeros(task.n)
    xis = np.random.randint(0, M, size=batch_size)

    for xi in xis:
        if xi == 0:
            g += M * task.grad_phi0(x, lam=lam)
        else:
            g += M * task.grad_phi_i(x, xi - 1)

    return g / batch_size, xis

In [ ]:
def custom_sgd_grad(task, x, lam=0.0):
    """
    Custom model:
        exact quadratic gradient + regularization
        + stochastic sampling over log-cosh terms only

    g(x, xi) = grad_phi0(x) + m * grad_phi_i(x)
    """
    i = np.random.randint(0, task.m)
    g = task.grad_phi0(x, lam=lam) + task.m * task.grad_phi_i(x, i)
    return g, i

In [ ]:
def custom_sgd_grad_minibatch(task, x, batch_size, lam=0.0):
    g = task.grad_phi0(x, lam=lam)
    idx = np.random.randint(0, task.m, size=batch_size)

    g_log = np.zeros(task.n)
    for i in idx:
        g_log += task.m * task.grad_phi_i(x, i)

    g += g_log / batch_size
    return g, idx

In [ ]:
def projected_sgd(task, x0, grad_sampler, step_rule, lam=0.0, eps=1e-2, max_iter=5000, keep_hist=False):
    x = task.project(x0.copy())
    iters = 0

    if keep_hist:
        hist_x = [x.copy()]
        hist_f = [task.f(x, lam)]
        hist_grad = [np.linalg.norm(task.grad(x, lam))]

    for k in range(max_iter):
        g_full = task.grad(x, lam)
        if np.linalg.norm(g_full) < eps:
            break

        h = step_rule(k)
        g_stoch, _ = grad_sampler(task, x, lam=lam)

        x = task.project(x - h * g_stoch)
        iters += 1

        if keep_hist:
            hist_x.append(x.copy())
            hist_f.append(task.f(x, lam))
            hist_grad.append(np.linalg.norm(task.grad(x, lam)))

    if keep_hist:
        return x, np.array(hist_x), np.array(hist_f), np.array(hist_grad), iters
    return x, iters

In [ ]:
def make_baseline_minibatch_sampler(batch_size):
    def sampler(task, x, lam=0.0):
        return baseline_sgd_grad_minibatch(task, x, batch_size=batch_size, lam=lam)
    return sampler

def make_custom_minibatch_sampler(batch_size):
    def sampler(task, x, lam=0.0):
        return custom_sgd_grad_minibatch(task, x, batch_size=batch_size, lam=lam)
    return sampler

In [ ]:
def const_step(h):
    return lambda k: h

def step_mu_over_k(mu_eff):
    return lambda k: 2.0 / (mu_eff * (k + 1))

def step_mu_over_sqrtk(mu_eff):
    return lambda k: 2.0 / (mu_eff * np.sqrt(k + 1))

In [ ]:
def projected_svrg_baseline(task, x0, gamma, epoch_length=50, lam=0.0, eps=1e-2, max_iter=5000, keep_hist=False):
    """
    SVRG for baseline decomposition:
        g = M * grad_phi_xi(x) - M * grad_phi_xi(x_tilde) + grad f(x_tilde)
    """
    x = task.project(x0.copy())
    iters = 0
    M = task.num_components_baseline()

    if keep_hist:
        hist_x = [x.copy()]
        hist_f = [task.f(x, lam)]
        hist_grad = [np.linalg.norm(task.grad(x, lam))]

    while iters < max_iter:
        x_tilde = x.copy()
        full_grad_tilde = task.grad(x_tilde, lam)

        for _ in range(epoch_length):
            if iters >= max_iter:
                break

            if np.linalg.norm(task.grad(x, lam)) < eps:
                if keep_hist:
                    return x, np.array(hist_x), np.array(hist_f), np.array(hist_grad), iters
                return x, iters

            xi = np.random.randint(0, M)

            if xi == 0:
                grad_x = M * task.grad_phi0(x, lam=lam)
                grad_xt = M * task.grad_phi0(x_tilde, lam=lam)
            else:
                grad_x = M * task.grad_phi_i(x, xi - 1)
                grad_xt = M * task.grad_phi_i(x_tilde, xi - 1)

            g = grad_x - grad_xt + full_grad_tilde

            x = task.project(x - gamma * g)
            iters += 1

            if keep_hist:
                hist_x.append(x.copy())
                hist_f.append(task.f(x, lam))
                hist_grad.append(np.linalg.norm(task.grad(x, lam)))

    if keep_hist:
        return x, np.array(hist_x), np.array(hist_f), np.array(hist_grad), iters
    return x, iters

In [ ]:
n = 20
t = Task(n=n, R=5.0)
x0 = np.random.randn(n)

# full projected GD
x_gd, it_gd = projected_gd(t, x0, gamma=1 / t.L(), eps=1e-2)
print("Projected GD:", it_gd, np.linalg.norm(t.grad(x_gd)))

# baseline SGD
mu_eff = 1e-1
x_sgd_base, it_sgd_base = projected_sgd(
    t, x0,
    grad_sampler=baseline_sgd_grad,
    step_rule=step_mu_over_sqrtk(mu_eff),
    eps=1e-2
)
print("Baseline SGD:", it_sgd_base, np.linalg.norm(t.grad(x_sgd_base)))

# custom SGD
x_sgd_custom, it_sgd_custom = projected_sgd(
    t, x0,
    grad_sampler=custom_sgd_grad,
    step_rule=step_mu_over_sqrtk(mu_eff),
    eps=1e-2
)
print("Custom SGD:", it_sgd_custom, np.linalg.norm(t.grad(x_sgd_custom)))

# SVRG
x_svrg, it_svrg = projected_svrg_baseline(
    t, x0,
    gamma=1 / (10 * t.L()),
    epoch_length=30,
    eps=1e-2
)
print("SVRG:", it_svrg, np.linalg.norm(t.grad(x_svrg)))

In [ ]:
t = Task(n=2, m=1, R=5.0)
x0 = np.array([-3.0, 3.0])

x_fin, hist_x, hist_f, hist_grad, iters = projected_gd(
    t, x0, gamma=1 / t.L(), eps=1e-3, keep_hist=True
)

Xg, Yg = np.meshgrid(np.linspace(-4, 4, 100), np.linspace(-4, 4, 100))
Z = np.zeros_like(Xg)

for i in range(Xg.shape[0]):
    for j in range(Xg.shape[1]):
        Z[i, j] = t.f(np.array([Xg[i, j], Yg[i, j]]))

plt.figure(figsize=(8, 6))
plt.contour(Xg, Yg, Z, levels=30)
plt.plot(hist_x[:, 0], hist_x[:, 1], 'ro-', markersize=4, label='Траектория')
plt.plot(hist_x[0, 0], hist_x[0, 1], 'k*', markersize=10, label='Старт')
plt.plot(hist_x[-1, 0], hist_x[-1, 1], 'g*', markersize=10, label='Финальная точка')
plt.legend()
plt.grid(True)
plt.title('Проецированный GD, n=2, m=1')
plt.show()